# RAG Application Development Notebook

This notebook demonstrates the complete development pipeline for building a Retrieval-Augmented Generation (RAG) system using LangChain, OpenAI, and FAISS.

## Table of Contents
1. [Environment Setup](#environment-setup)
2. [Data Ingestion](#data-ingestion)
3. [Text Chunking](#text-chunking)
4. [Vector Store Creation](#vector-store-creation)
5. [Similarity Search](#similarity-search)
6. [RAG Chain Construction](#rag-chain-construction)
7. [Testing the System](#testing-the-system)

## 1. Environment Setup

Install required dependencies and load environment variables.

In [ ]:
# Install required packages
! pip install langchain openai tiktoken rapidocr-onnxruntime python-dotenv langchain-community faiss-cpu

In [ ]:
# Load environment variables
import os
from dotenv import load_dotenv

load_dotenv()
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

## 2. Data Ingestion

Load documents from a text file using LangChain's `TextLoader`.

In [ ]:
from langchain.document_loaders import TextLoader

# Load the document
loader = TextLoader("data/Agentic AI.txt", encoding="utf8")
documents = loader.load()

# Preview the first 500 characters
print(documents[0].page_content[:500])

## 3. Text Chunking

Split documents into smaller chunks for better retrieval performance.

In [ ]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

# Initialize text splitter with chunk size and overlap
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=200,
    chunk_overlap=20
)

# Split documents into chunks
text_chunks = text_splitter.split_documents(documents)

print(f"Total chunks created: {len(text_chunks)}")
print(f"\nFirst chunk preview:\n{text_chunks[0].page_content}")

## 4. Vector Store Creation

Create embeddings and build a FAISS vector store for efficient similarity search.

In [ ]:
from langchain.embeddings import OpenAIEmbeddings
from langchain.vectorstores import FAISS

# Initialize embeddings model
embeddings = OpenAIEmbeddings()

# Create FAISS vector store from documents
vectorstore = FAISS.from_documents(text_chunks, embeddings)

print(f"Vector store created: {vectorstore}")

## 5. Similarity Search

Test the vector store by performing similarity search.

In [ ]:
# Perform similarity search
query = "What are the key characteristics of Agentic AI?"
docs = vectorstore.similarity_search(query, k=4)

# Display the results
for i, doc in enumerate(docs):
    print(f"\n{'='*60}")
    print(f"Document {i+1}:")
    print(f"{'='*60}")
    print(doc.page_content)

## 6. RAG Chain Construction

Build the complete RAG pipeline with prompt template, LLM, and output parser.

In [ ]:
from langchain.prompts import ChatPromptTemplate
from langchain.schema.output_parser import StrOutputParser
from langchain.chat_models import ChatOpenAI
from langchain.schema.runnable import RunnablePassthrough

# Define the prompt template
template = """You are an assistant for question-answering tasks.
Use the following pieces of retrieved context to answer the question.
If you don't know the answer, just say that you don't know.
Use ten sentences maximum and keep the answer concise.

Question: {question}
Context: {context}

Answer:
"""

# Create prompt from template
prompt = ChatPromptTemplate.from_template(template)

# Initialize LLM
llm_model = ChatOpenAI(model_name="gpt-4o-mini")

# Initialize output parser
output_parser = StrOutputParser()

# Create retriever from vector store
retriever = vectorstore.as_retriever()

# Build the RAG chain
rag_chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt
    | llm_model
    | output_parser
)

print("RAG chain successfully created!")

## 7. Testing the System

Test the complete RAG pipeline with sample queries.

In [ ]:
# Test Query 1: General overview
query1 = "Tell me about Agentic AI"
response1 = rag_chain.invoke(query1)

print(f"Query: {query1}")
print(f"\nResponse:\n{response1}")

In [ ]:
# Test Query 2: Specific characteristics
query2 = "What are the key characteristics of Agentic AI?"
response2 = rag_chain.invoke(query2)

print(f"Query: {query2}")
print(f"\nResponse:\n{response2}")

In [ ]:
# Test Query 3: Applications
query3 = "What are the applications of Agentic AI?"
response3 = rag_chain.invoke(query3)

print(f"Query: {query3}")
print(f"\nResponse:\n{response3}")

## Summary

This notebook demonstrated:
- Loading and processing documents
- Creating vector embeddings with OpenAI
- Building a FAISS vector store
- Constructing a complete RAG pipeline
- Testing with various queries

### Next Steps
- Experiment with different chunk sizes and overlap values
- Try different embedding models
- Test with multiple documents
- Implement conversation memory for multi-turn dialogues
- Evaluate retrieval quality and answer accuracy